# Unsupervised Learning: t-SNE Dimensionality Reduction

---

## Introduction

**t-SNE** (t-distributed Stochastic Neighbour Embedding) is a dimensionality reduction technique designed specifically for **visualization**. It maps high-dimensional data to 2D or 3D while preserving the local neighbourhood structure — samples that are close in the original space remain close in the projection.

### t-SNE vs. PCA

| Property | PCA | t-SNE |
|---|---|---|
| Type | Linear | Non-linear |
| Preserves | Global variance | Local neighbourhood structure |
| Interpretable axes | Yes (principal components) | No |
| Deterministic | Yes | No (random seed needed) |
| Speed | Fast | Slow on large data |
| Best for | Preprocessing, compression | Visualization only |

### Key Hyperparameter — `perplexity`

`perplexity` controls the balance between local and global structure — it roughly corresponds to the number of nearest neighbours each point considers. Typical values: 5–50. Lower perplexity preserves fine local structure; higher perplexity captures broader patterns.

This notebook applies t-SNE to the **MNIST handwritten digits** dataset (784-dimensional images → 2D) and the **Iris dataset** for a cleaner comparison.

---
## 1. Imports

In [ ]:
!pip uninstall matplotlib -y

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.manifold import TSNE
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_digits, load_iris, fetch_openml

import warnings
warnings.filterwarnings('ignore')


---
## 2. t-SNE on Iris Dataset

We start with Iris (4D → 2D) because its ground-truth labels let us validate visually that t-SNE correctly separates the three species.

In [ ]:
iris = load_iris()
X_iris = StandardScaler().fit_transform(iris.data)
y_iris = iris.target
species = ['setosa', 'versicolor', 'virginica']

# t-SNE projection
tsne_iris = TSNE(n_components=2, perplexity=30, random_state=42)
X_iris_2d = tsne_iris.fit_transform(X_iris)

# PCA for comparison
pca_iris = PCA(n_components=2)
X_iris_pca = pca_iris.fit_transform(X_iris)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, X_proj, title in zip(
    axes,
    [X_iris_pca, X_iris_2d],
    ['PCA (linear)', 't-SNE (non-linear)']
):
    for i, name in enumerate(species):
        mask = y_iris == i
        ax.scatter(X_proj[mask, 0], X_proj[mask, 1],
                   label=name, alpha=0.8, edgecolors='k', linewidths=0.3, s=40)
    ax.set_title(f'Iris — {title}')
    ax.legend()
    ax.set_xlabel('Dim 1')
    ax.set_ylabel('Dim 2')

plt.suptitle('PCA vs. t-SNE on Iris (4D → 2D)', fontsize=13)
plt.tight_layout()
plt.show()

---
## 3. t-SNE on MNIST Digits

MNIST contains 70,000 handwritten digit images (28×28 = 784 features). We use sklearn's built-in `load_digits` (1797 samples, 64 features) for speed, then demonstrate on MNIST-784 with a 1000-sample subset.

In [ ]:
# sklearn digits (fast — 64 features)
digits = load_digits()
X_digits = StandardScaler().fit_transform(digits.data)
y_digits = digits.target

print('Digits dataset shape:', X_digits.shape)
print('Classes:', np.unique(y_digits))

tsne_digits = TSNE(n_components=2, perplexity=30, random_state=42)
X_digits_2d = tsne_digits.fit_transform(X_digits)

plt.figure(figsize=(9, 7))
scatter = plt.scatter(X_digits_2d[:, 0], X_digits_2d[:, 1],
                      c=y_digits, cmap='tab10', alpha=0.7,
                      edgecolors='k', linewidths=0.2, s=25)
plt.colorbar(scatter, label='Digit Class')
plt.title('t-SNE — sklearn Digits (64D → 2D)')
plt.xlabel('Dim 1')
plt.ylabel('Dim 2')
plt.tight_layout()
plt.show()

---
## 4. t-SNE on MNIST-784 (1000 samples)

For the full 784-feature MNIST, we use 1000 samples and apply PCA first (to 50 components) before t-SNE — a common practice to speed up computation while preserving most variance.

In [ ]:
# Fetch MNIST-784
mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X_mnist = mnist.data[:1000].astype(float)
y_mnist = mnist.target[:1000]

# Standardize + PCA pre-reduction → t-SNE
X_scaled = StandardScaler().fit_transform(X_mnist)
X_pca50  = PCA(n_components=50, random_state=42).fit_transform(X_scaled)
X_tsne   = TSNE(n_components=2, perplexity=30, random_state=42).fit_transform(X_pca50)

print('Shape after PCA:', X_pca50.shape)
print('Shape after t-SNE:', X_tsne.shape)

tsne_df = pd.DataFrame({'Dim_1': X_tsne[:, 0], 'Dim_2': X_tsne[:, 1], 'label': y_mnist})

plt.figure(figsize=(9, 7))
sns.scatterplot(data=tsne_df, x='Dim_1', y='Dim_2',
                hue='label', palette='bright', s=25, alpha=0.8,
                edgecolor='k', linewidth=0.2)
plt.title('t-SNE — MNIST-784 (1000 samples, PCA→50D→2D)')
plt.xlabel('Dim 1')
plt.ylabel('Dim 2')
plt.legend(title='Digit', bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

---
## 5. Effect of Perplexity

In [ ]:
perplexities = [5, 15, 30, 50]

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, p in zip(axes, perplexities):
    proj = TSNE(n_components=2, perplexity=p, random_state=42).fit_transform(X_digits)
    ax.scatter(proj[:, 0], proj[:, 1], c=y_digits, cmap='tab10',
               alpha=0.7, edgecolors='k', linewidths=0.1, s=15)
    ax.set_title(f'perplexity={p}', fontsize=10)
    ax.set_xlabel('Dim 1')

plt.suptitle('t-SNE — Effect of Perplexity on Digit Dataset', fontsize=13)
plt.tight_layout()
plt.show()

---
## Conclusion

- **t-SNE on Iris** produces cleaner class separation than PCA, showing that the non-linear structure between versicolor and virginica is better captured by t-SNE's local neighbourhood preservation.
- **t-SNE on sklearn digits** (64D → 2D) reveals 10 well-separated clusters corresponding to digits 0–9 — a strong visual confirmation that the feature space contains meaningful structure.
- **MNIST-784** (784D → 50D → 2D) demonstrates the recommended practice: apply PCA first to reduce noise and speed up t-SNE, then run t-SNE on the compressed representation.
- The **perplexity comparison** shows that very low perplexity (5) creates tight, isolated islands while higher values (30–50) produce more globally coherent layouts. Values of 20–40 are generally a good starting range.
- **Important**: t-SNE is for visualization only — distances and cluster sizes in the t-SNE plot are not directly interpretable as they are in PCA. Never use t-SNE outputs as features for downstream models.